# 🚀 Welcome to **Ultra**

> A streamlined backtesting environment for rapid model iteration and evaluation.

---

## Overview

**Ultra** is designed to efficiently backtest:

- 📊 **Existing models** — validate current production models against historical data
- 🔧 **Model modifications** — test tweaks and improvements before deployment
- 🧪 **New variables** — evaluate feature additions and their predictive power

---
| Feature | Description |
|---------|-------------|
| ⚡ **Production-level accuracy** | Uses production datasets, methods, and node selection that closely mirror the live environment |
| 🎯 **Compact node selection** | For initial backtests, focuses on a rotating subset — selecting a few nodes per day while ensuring the full training set covers all unique nodes |
| 📈 **Fast visualization** | Leverages Kubernetes to run each backtest within minutes, enabling rapid visualization of results |
| 🔄 **Two-level backtest** | Runs an initial backtest on a smaller subset first; if significant improvement is observed, it escalates to a full backtest |
---

## Testing Content

**Guide:** We benchmark all results against the **Fourier model**, keeping every variable constant — testing window, node selection, and all other parameters — except for the specific model structure or variable being tested.

Each experiment is graded as follows:

| Grade | Meaning |
|:-----:|---------|
| 🟢 **S** | Significant improvement over production |
| 🔵 **A** | Slight improvement, but not significant |
| ⚪ **B** | Nearly identical to production |
| 🟡 **C** | Slightly worse than production |
| 🔴 **F** | Significantly worse than production (failed) |

---

### Model Tweaks

1. Fourier with training set, no shuffle — **B**
2. Fourier with larger batch size: **C**
2. CNN + Fourier: **F**
3. Fourier + CNN: **F**
4. pure CNN: **F**
5. GRU: **S**
6. GRU with updated parameters(hidden size: 16, learning rate: 0.001, out= mean(), better than 5th): **S**

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '/var/www/python/Qingcheng/Darwin')
sys.path.append('/var/www/python/Prod/nighthawk/')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections
from nighthawk.models.valuation import node_price_predictor
import time
import pickle
import dill
from google.cloud import bigquery, storage
warnings.filterwarnings("ignore")
from datetime import datetime
from nighthawk.models.valuation import ve_model_functions
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sys, os, inspect, shutil, importlib
sys.path.insert(0, '/var/www/python/Qingcheng/Fourier')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections, sql_functions, dataframe_functions
from nighthawk.data.pipeline.var_handler import loadwindgen_vh, wind_vh
from nighthawk.data.pipeline.common_functions import wind
from google.cloud import bigquery
import utils_fourier.ve_portfolio_constructor_fourier as ve_portfolio_constructor_fourier
from common_functions import get_scale_factor
from nighthawk.data.network.node import Node
sys.path.insert(0, '/var/www/python/Qingcheng/QCTest/Ultra')
from custom import get_metrics, eval_valuation_model, run_in_kubernetes, fourier_port, simulate_total_ftp, select_unique_nodes_across_dates, pnl_metrics, run_valuation_backtest
from model import GRU_framework_means_update_para, GRU_framework_means_print
import math


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
result_with_noshuffle, valuation_result_noshuffle = run_valuation_backtest(GRU_framework_means_print, 5)

  record_id 50150
We gonna upload data into cloud sql!
data upload done!
docker: 'docker rmi' requires at least 1 argument

Usage:  docker rmi [OPTIONS] IMAGE [IMAGE...]

See 'docker rmi --help' for more information

Login Succeeded
 
WARNING! Your credentials are stored unencrypted in '/home/qingcheng/.docker/config.json'.
Configure a credential helper to remove this warning. See
https://docs.docker.com/go/credential-store/


Using default tag: latest
latest: Pulling from movetocloud-999/fourier/ve_2024_nn
Digest: sha256:22c06995af4e5d82cc39deb8db214c0f8193dd36eef898519bacea228b671c5a
Status: Image is up to date for us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
 
 #0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 382B done
#1 DONE 0.1s

#2 [internal] load metadata for us-central1-docker.pkg.dev/movetocloud-99

KeyboardInterrupt: 

In [ ]:
50063

hidden size: 32, learning rate: 0.01, out = last one 
da_total_ME      -5.03    
da_total_MSE    411.89
da_total_MAE     14.98
da_total_RMSE    20.29
rt_total_ME      -2.79
rt_total_MSE   3760.65
rt_total_MAE     25.75
rt_total_RMSE    61.32

hidden size: 16, learning rate: 0.001, out = mean(), refer to 50064
da_total_ME      -5.39
da_total_MSE    377.39
da_total_MAE     14.62
da_total_RMSE    19.43
rt_total_ME      -2.06
rt_total_MSE   3746.30
rt_total_MAE     25.88
rt_total_RMSE    61.21

Full backtest for 273 for GRU on updated parameter
da_total_ME      -5.73
da_total_MSE   2254.18
da_total_MAE     21.69
da_total_RMSE    47.48
rt_total_ME      -0.08
rt_total_MSE   3434.03
rt_total_MAE     26.89
rt_total_RMSE    58.60


In [4]:
import pandas as pd
nodes = pd.read_csv('/var/www/python/Qingcheng/WFiles/Ultra/node_selection_42_0121_0421.csv')
display(nodes[:20])

,Unnamed: 0,dt,node_num
0,0,2026-01-21,1217
1,1,2026-01-21,229
2,2,2026-01-21,1384
3,3,2026-01-22,181
4,4,2026-01-22,1185
5,5,2026-01-22,1524
6,6,2026-01-23,931
7,7,2026-01-23,1683
8,8,2026-01-23,1295
9,9,2026-01-24,917


In [6]:
import pandas as pd

df931  = pd.read_csv('gs://ve_fourier/production/SPP/training/1499_2026-01-26.csv', nrows=1)
df1683 = pd.read_csv('gs://ve_fourier/production/SPP/training/982_2026-01-26.csv', nrows=1)

cols931  = set(df931.columns)
cols1683 = set(df1683.columns)

only_931  = sorted(cols931 - cols1683)
only_1683 = sorted(cols1683 - cols931)
shared    = sorted(cols931 & cols1683)

print(f'931 total cols:  {len(cols931)}')
print(f'1683 total cols: {len(cols1683)}')
print(f'Shared: {len(shared)}')
print(f'\nOnly in 931  ({len(only_931)}):  {only_931}')
print(f'\nOnly in 1683 ({len(only_1683)}): {only_1683}')
print(f'\nAll 931 columns ({len(cols931)}):')
for c in sorted(df931.columns):
    print(' ', c)


931 total cols:  239
1683 total cols: 237
Shared: 185

Only in 931  (54):  ['BackwardRampNoSlope1_okge_spp_zonal_load_actual_a', 'BackwardRampNoSlope1_okge_spp_zonal_load_forecast_f', 'ForwardRampNoSlope1_okge_spp_zonal_load_actual_a', 'ForwardRampNoSlope1_okge_spp_zonal_load_forecast_f', 'Perc_30D_okge_spp_zonal_load_actual_a', 'Perc_30D_okge_spp_zonal_load_forecast_f', 'amarillo_tx_159_appt_degf', 'amarillo_tx_159_temperature_degf', 'amarillo_tx_159_temperature_degf_window_zscore', 'amarillo_tx_159_temperature_humidity_index', 'amarillo_tx_159_windspeed', 'eai_miso_zonal_load_actual_a', 'eai_miso_zonal_load_forecast_f', 'ees_miso_zonal_load_actual_a', 'ees_miso_zonal_load_forecast_f', 'gfs_wspd80_952_SolomonForksWin_actual_a', 'gfs_wspd80_952_SolomonForksWin_forecast_f', 'la_statelevel_cdd_actual_a', 'la_statelevel_cdd_forecast_f', 'la_statelevel_cdd_window_zscore_enhanced_actual_a', 'la_statelevel_cdd_window_zscore_enhanced_forecast_f', 'la_statelevel_hdd_actual_a', 'la_statelevel_h